In [1]:
import pandas as pd
import numpy as np
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix

from xgboost import XGBClassifier
from scipy.sparse import hstack

sys.path.append(os.path.abspath(".."))

In [2]:
df = pd.read_csv("../data/raw/credit_dataset.csv")

In [3]:
def generate_location_text(row):
    if row["income"] > 80000 and row["credit_score"] > 680:
        return "Affluent urban residential area"
    elif row["income"] > 50000:
        return "Urban middle income locality"
    elif row["income"] > 30000:
        return "Tier 2 service economy town"
    else:
        return "Semi rural low income region"

df["location_text"] = df.apply(generate_location_text, axis=1)

In [4]:
def generate_officer_notes(row):
    if row["debt_to_income"] > 0.45:
        return "High debt burden relative to income"
    elif row["credit_score"] < 600:
        return "Past credit issues observed"
    elif row["loan_amount"] > row["income"]:
        return "Loan amount high relative to income"
    else:
        return "No major risk factors identified"

df["officer_notes"] = df.apply(generate_officer_notes, axis=1)

In [8]:
export_cols = [
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount",
    "location_text",
    "officer_notes",
    "approved"
]

df[export_cols].to_csv(
    "../data/processed/credit_tab_text_conditioned.csv",
    index=False
)

In [9]:
X_tabular = df[[
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount"
]]

y = df["approved"]

In [10]:
X_train_tab, X_test_tab, y_train, y_test = train_test_split(
    X_tabular,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [11]:
tfidf_location = TfidfVectorizer(
    max_features=10,
    stop_words="english"
)

X_train_loc = tfidf_location.fit_transform(
    df.loc[X_train_tab.index, "location_text"]
)
X_test_loc = tfidf_location.transform(
    df.loc[X_test_tab.index, "location_text"]
)


tfidf_notes = TfidfVectorizer(
    max_features=20,
    stop_words="english"
)

X_train_notes = tfidf_notes.fit_transform(
    df.loc[X_train_tab.index, "officer_notes"]
)
X_test_notes = tfidf_notes.transform(
    df.loc[X_test_tab.index, "officer_notes"]
)

In [12]:
X_train = hstack([
    X_train_tab.values,
    X_train_loc,
    X_train_notes
])

X_test = hstack([
    X_test_tab.values,
    X_test_loc,
    X_test_notes
])

In [13]:
cost_fp = 100
cost_fn = 10

sample_weights = np.where(y_train == 0, cost_fp, cost_fn)

In [14]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
y_pred = xgb.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

expected_cost = FP * cost_fp + FN * cost_fn
cost_per_applicant = expected_cost / len(y_test)

cm, expected_cost, cost_per_applicant

(array([[1156,   44],
        [ 428, 1372]]),
 8680,
 2.8933333333333335)